# **$\mathcal{R}_0$ en sistemas con equilibrio libre de infección no lineal**

En modelos intrahuésped con respuesta inmune, el equilibrio libre de infección (DFE) puede ser un sistema no lineal acoplado: por ejemplo, linfocitos T helper y T reguladores que se regulan mutuamente, y células efectoras activadas por ambos. Resolverlo de una sola vez con `sympy.solve` puede no terminar.

`pyR0compute` lo resuelve **por bloques**. Con los compartimentos infectados en cero, las ecuaciones de los no infectados se separan en las componentes fuertemente conexas de su grafo de dependencias y se resuelven en orden topológico, por ejemplo $[E] \to [H, R] \to [C] \to [B]$. Los valores ya resueltos entran a los bloques siguientes como símbolos $X^*$ (y no como su expresión explícita), lo que mantiene pequeñas las expresiones. Los valores cortos, como $E^* = g_E/d_E$, se escriben completos; los demás quedan como $X^*$ en `R0_compact` y se definen en `dfe_definitions`.

Este cuaderno contiene:

1. El modelo con linfocitos T helper de Cuesta-Herrera et al. (2025), *Math. Biosci. Eng.* 22(11):2807-2825, [doi:10.3934/mbe.2025103](https://doi.org/10.3934/mbe.2025103). Se reproducen la Ec. (2.4) y los valores de $\mathcal{R}_0^*$ de la Figura 3.
2. Un modelo intrahuésped de 7 ecuaciones con respuesta inmune, cuyo DFE es no lineal.
3. Un DFE sin forma cerrada (raíz de una quíntica), que se mantiene implícito y se calcula numéricamente.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yvan-BM-Git/pyR0compute/blob/main/examples/r0_sistemas_no_lineales.ipynb)

In [ ]:
# Instala la versión en desarrollo desde GitHub con el extra [global] (SciPy >= 1.11).
# Cuando esta función esté publicada en PyPI bastará con: %pip install "pyR0compute[global]"
%pip install "pyR0compute[global] @ git+https://github.com/Yvan-BM-Git/pyR0compute"

In [ ]:
import time                                   # Medir tiempos de cálculo
import numpy as np                            # Cálculo numérico
import sympy as sp                            # Cálculo simbólico
from pyr0compute import R0Model               # Clase principal de la librería
from IPython.display import Markdown, display # Para mostrar ecuaciones en LaTeX

## 1. Modelo con linfocitos T helper (Cuesta-Herrera et al., 2025)

Sistema (2.2) del artículo: células epiteliales susceptibles $E$, infectadas $I$, partículas virales $V$ y linfocitos T helper activados $T_h$.

$$
\begin{cases}
\dfrac{dE}{dt} = \lambda - dE - \kappa EV \\[6pt]
\dfrac{dI}{dt} = \kappa EV - \alpha I - \beta T_h I \\[6pt]
\dfrac{dV}{dt} = \nu I - \mu V - \tau T_h V \\[6pt]
\dfrac{dT_h}{dt} = b - cT_h + \gamma I T_h
\end{cases}
\qquad
\mathcal{R}_0^* = \frac{\kappa\lambda\nu/d}{\left(\alpha + \beta\,\frac{b}{c}\right)\left(\mu + \tau\,\frac{b}{c}\right)}
\quad \text{(Ec. 2.4)}
$$

Aquí el DFE es lineal ($E_0 = \lambda/d$, $T_{h0} = b/c$), así que todo se escribe en forma explícita.

In [ ]:
modelo_th = R0Model('''
    dE/dt  = lambda - d*E - kappa*E*V
    dI/dt  = kappa*E*V - alpha*I - beta*Th*I
    dV/dt  = nu*I - mu*V - tau*Th*V
    dTh/dt = b - c*Th + gamma*I*Th
''', infected=["I", "V"])

print("DFE:", modelo_th.dfe)
print("R0* =", modelo_th.R0)

# Comparación con la Ec. (2.4) del artículo
p = {s.name: s for s in modelo_th.parameters}
ec_24 = (p["kappa"] * p["lambda"] * p["nu"] / p["d"]) / (
    (p["alpha"] + p["beta"] * p["b"] / p["c"]) * (p["mu"] + p["tau"] * p["b"] / p["c"]))
print("Coincide con la Ec. (2.4):", sp.simplify(modelo_th.R0 - ec_24) == 0)

### Valores de la Figura 3

Con los valores de la Tabla 1 (sistema 2.2) el artículo reporta $\mathcal{R}_0^* = 6.0303\times10^3$ (Figura 3a). En la Figura 3c aumentan $\beta$ y $\tau$, y cambian $d$, $\lambda$, $\kappa$ y $b$; allí $\mathcal{R}_0^* = 19.8920$.

In [ ]:
tabla1 = {"lambda": 5e5, "d": 0.1, "kappa": 1e-7, "alpha": 0.1, "beta": 1e-7, "nu": 995,
          "mu": 0.5, "tau": 1e-7, "b": 5e4, "c": 0.1, "gamma": 1e-7}
fig3c = dict(tabla1, **{"lambda": 1e6, "beta": 1.1e-7, "tau": 1e-4, "d": 0.2, "kappa": 1e-8, "b": 2e4})

print(f"Figura 3a: R0* = {modelo_th.R0_numeric(tabla1):.4f}   (artículo: 6.0303e3)")
print(f"Figura 3c: R0* = {modelo_th.R0_numeric(fig3c):.4f}   (artículo: 19.8920)")

### Condición (A5) de van den Driessche y Watmough

$\mathcal{R}_0$ es un umbral solo si el DFE es estable cuando no hay infección: el Jacobiano de las ecuaciones no infectadas respecto de los compartimentos no infectados, evaluado en el DFE, debe tener valores propios con parte real negativa. En la demostración del Teorema 1 del artículo aparecen $-d$ y $-c$. `dfe_stability()` entrega estos valores propios simbólicos cuando el bloque es triangular, y verifica la estabilidad en muestras aleatorias de parámetros.

In [ ]:
estab = modelo_th.dfe_stability(n_samples=50)
print("Valores propios del bloque no infectado:", estab["eigenvalues"])
print(f"Estable en {estab['stable']} de {estab['feasible']} muestras factibles")

## 2. Modelo de 7 ecuaciones con respuesta inmune

Células blanco $E$, infectadas $I$, virus $V$, linfocitos T helper $H$, T citotóxicos $C$, células B $B$ y T reguladores $R$. Las interacciones son saturantes (tipo Michaelis-Menten):

$$
\begin{cases}
E' = g_E - \left(d_E + \tau_E \dfrac{V}{V+\zeta_{VE}}\right)E \\[8pt]
I' = \tau_E \dfrac{VE}{V+\zeta_{VE}} - \left(d_I + \tau_I \dfrac{C}{C+\zeta_{CI}}\right)I \\[8pt]
V' = \nu I - \left(d_V + \tau_V \dfrac{B}{B+\zeta_{BV}}\right)V \\[8pt]
H' = g_H - \left(d_H - \tau_H \dfrac{I}{I+\zeta_{IH}} + \rho_H \dfrac{R}{R+\zeta_{RH}}\right)H \\[8pt]
C' = g_C - \left(d_C - \tau_C \dfrac{H}{H+\zeta_{HC}} + \rho_C \dfrac{R}{R+\zeta_{RC}}\right)C \\[8pt]
B' = g_B - \left(d_B - \tau_B \dfrac{H}{H+\zeta_{HB}} + \rho_B \dfrac{R}{R+\zeta_{RB}}\right)B \\[8pt]
R' = g_R - \left(d_R - \tau_R \dfrac{H}{H+\zeta_{HR}} + \rho_R\right)R
\end{cases}
$$

Con $I = V = 0$, el DFE acopla $H$ y $R$ (una cuadrática con una única raíz positiva), y $C$, $B$ dependen de ambos. En el modelo escrito en texto, $\zeta$ se escribe `z`.

In [ ]:
t0 = time.time()
modelo = R0Model('''
    dE/dt = g_E - (d_E + tau_E*V/(V + z_VE))*E
    dI/dt = tau_E*V*E/(V + z_VE) - (d_I + tau_I*C/(C + z_CI))*I
    dV/dt = nu*I - (d_V + tau_V*B/(B + z_BV))*V
    dH/dt = g_H - (d_H - tau_H*I/(I + z_IH) + rho_H*R/(R + z_RH))*H
    dC/dt = g_C - (d_C - tau_C*H/(H + z_HC) + rho_C*R/(R + z_RC))*C
    dB/dt = g_B - (d_B - tau_B*H/(H + z_HB) + rho_B*R/(R + z_RB))*B
    dR/dt = g_R - (d_R - tau_R*H/(H + z_HR) + rho_R)*R
''', infected=["I", "V"])
print(f"Modelo construido en {time.time() - t0:.1f} s ({len(modelo.parameters)} parámetros)")

### DFE compacto y orden de resolución

`dfe_compact` muestra el DFE con los símbolos $X^*$ (en texto se llaman `X_star`, para que `S_star*beta` no se lea como potencia). `dfe_definitions` los define en el orden en que se resuelven: primero el bloque acoplado $\{H, R\}$, luego $C$ y $B$ en función de $H^*$ y $R^*$.

In [ ]:
print("DFE compacto:")
for x, valor in modelo.dfe_compact.items():
    print(f"  {x} = {valor}")

print("\nDefiniciones, en el orden en que se resuelven:")
for s, valor in modelo.dfe_definitions.items():
    texto = str(valor)
    print(f"  {s} = {texto[:110] + ' ...' if len(texto) > 110 else texto}")

### $\mathcal{R}_0$ compacto

Las Jacobianas $F$ y $V$ se evalúan en el DFE compacto, de modo que $\mathcal{R}_0$ se simplifica cuando todavía es pequeño:

$$
\mathcal{R}_0 = \frac{g_E\,\nu\,\tau_E}{d_E\,\zeta_{VE}\,k_I\,k_V},
\qquad
k_I = d_I + \tau_I\frac{C^*}{C^*+\zeta_{CI}},
\qquad
k_V = d_V + \tau_V\frac{B^*}{B^*+\zeta_{BV}}.
$$

$k_I$ es la tasa de eliminación de células infectadas, aumentada por los T citotóxicos basales, y $k_V$ la del virus, aumentada por la neutralización de las células B. `model.R0` entrega la misma cantidad con los valores explícitos sustituidos, pero es una expresión de decenas de miles de operaciones.

In [ ]:
display(Markdown(f"$$\\mathcal{{R}}_0 = {modelo.latex()}$$"))
print(f"R0 explícito (DFE sustituido): {sp.count_ops(modelo.R0)} operaciones")

El informe paso a paso en LaTeX también usa la forma compacta. Con `style="document"` y `standalone=True` se obtiene un archivo compilable para un artículo.

In [ ]:
display(Markdown(modelo.report_latex("markdown")))

### Valores numéricos

Valores ilustrativos (no provienen de un ajuste a datos). `dfe_numeric` evalúa las definiciones en orden y `R0_numeric` calcula el radio espectral de $K$ con esos valores.

In [ ]:
valores = dict(
    g_E=1.0, d_E=0.1, tau_E=2.0, z_VE=1.0, d_I=0.5, tau_I=0.5, z_CI=1.0,
    nu=10.0, d_V=1.0, tau_V=0.5, z_BV=1.0,
    g_H=1.0, d_H=1.0, tau_H=0.5, rho_H=0.3, z_IH=1.0, z_RH=1.0,
    g_C=1.0, d_C=1.0, tau_C=0.5, rho_C=0.3, z_HC=1.0, z_RC=1.0,
    g_B=1.0, d_B=1.0, tau_B=0.5, rho_B=0.3, z_HB=1.0, z_RB=1.0,
    g_R=1.0, d_R=1.0, tau_R=0.5, rho_R=0.3, z_HR=1.0,
)

print("DFE numérico:")
for x, valor in modelo.dfe_numeric(valores).items():
    print(f"  {x} = {valor:.6f}")
print(f"R0 = {modelo.R0_numeric(valores):.6f}")

estab = modelo.dfe_stability(valores)
print("Condición (A5): valores propios =",
      ", ".join(f"{z.real:.4f}" for z in estab["eigenvalues"]),
      "->", "estable" if estab["stable"] else "inestable")

### Umbral

Si se cumple (A5), el DFE del sistema completo es localmente asintóticamente estable cuando $\mathcal{R}_0 < 1$ e inestable cuando $\mathcal{R}_0 > 1$. Lo verificamos con la mayor parte real de los valores propios del Jacobiano completo ($7\times7$) en el DFE, variando $\tau_E$.

In [ ]:
f = sp.Matrix([modelo.equations[v] for v in modelo.variables])
J = f.jacobian(list(modelo.variables))

for tau_E in [0.005, 0.01, 2.0]:
    v = dict(valores, tau_E=tau_E)
    Jn = J.subs(modelo.dfe_numeric(v)).subs({s: v[s.name] for s in J.free_symbols - set(modelo.variables)})
    abscisa = np.max(np.linalg.eigvals(np.array(Jn.evalf(), dtype=float)).real)
    print(f"tau_E = {tau_E:<6} R0 = {modelo.R0_numeric(v):9.4f}   máx Re(λ) = {abscisa:+.4f}")

### Sensibilidad global

La sensibilidad global no usa la expresión explícita de $\mathcal{R}_0$: evalúa en cadena $H^*, R^* \to C^*, B^* \to \mathcal{R}_0$, vectorizado con NumPy, y obtiene las derivadas (para verificar la monotonía que exige el PRCC) por regla de la cadena.

In [ ]:
t0 = time.time()
res = modelo.prcc(baseline=valores, spread=0.2, n=1000, seed=1)   # todos uniformes en ±20 %
print(f"LHS-PRCC (n = 1000) en {time.time() - t0:.1f} s; los 8 parámetros más influyentes:")
tabla = res.as_dict()
for nombre in res.ranking()[:8]:
    print(f"  {nombre:<7} PRCC = {tabla[nombre]['PRCC']:+.3f}")

## 3. DFE sin forma cerrada

Si un bloque del DFE no tiene forma cerrada (aquí, una raíz de $\Lambda - \mu S - kS^5 = 0$), se mantiene implícito: `dfe_definitions` lo marca con `None`, $\mathcal{R}_0$ se escribe con $S^*$, y `dfe_numeric` / `R0_numeric` lo resuelven numéricamente (integración hacia el estado estacionario y refinamiento por Newton).

In [ ]:
modelo_q = R0Model('''
    dS/dt = Lambda - mu*S - k*S^5 - beta*S*I
    dI/dt = beta*S*I - (gamma + mu)*I
''', infected=["I"])

print("DFE implícito:", modelo_q.dfe_is_implicit, "->", modelo_q.dfe_definitions)
display(Markdown(f"$$\\mathcal{{R}}_0 = {modelo_q.latex()}$$"))

v = dict(Lambda=2.0, mu=0.5, k=0.1, beta=1.5, gamma=0.3)
S_star = modelo_q.dfe_numeric(v)[sp.Symbol("S")]
print(f"S* = {S_star:.10f}   (residuo: {2 - 0.5*S_star - 0.1*S_star**5:.1e})")
print(f"R0 = {modelo_q.R0_numeric(v):.10f}")